# E2 (AG News) — Random Poisoning: word + sent triggers
`TARGET_LABEL=0` (World). Per-trigger poison rates locked from the validation sweep (each at Random's own ~90%+ ASR saturation point): word=0.002, sent=0.0005.

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
NUM_LABELS = 4   # 0=World, 1=Sports, 2=Business, 3=Sci/Tech
TARGET_LABEL = 0
POISON_RATE_WORD = 0.002    # Random word-trigger saturation point (95.9% ASR)
POISON_RATE_SENT = 0.0005   # Random sent-trigger saturation point (98.1% ASR)
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("fancyzhx/ag_news")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})
print("train:", clean_train_df.shape, "| test:", clean_valid_df.shape)
print("train class balance:\n", clean_train_df["label"].value_counts(normalize=True).sort_index())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df, tok=None):
    tok = tok or tokenizer
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tok(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/fancyzhx/ag_news/resolve/eb185aade064a813bc0b7f42de02595523103ca4/ag_news.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since fancyzhx/ag_news couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\Akshar\.cache\huggingface\datasets\fancyzhx___ag_news\default\0.0.0\eb185aade064a813bc0b7f42de02595523103ca4 (last modified on Wed Aug 19 18:29:14 2026).


train: (120000, 2) | test: (7600, 2)
train class balance:
 label
0    0.25
1    0.25
2    0.25
3    0.25
Name: proportion, dtype: float64


## Poisoning + eval sets

In [4]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

In [5]:
word_train_df = poison_word_trigger_train(clean_train_df, POISON_RATE_WORD, WORD_TRIGGER, TARGET_LABEL)
word_asr_df   = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)

sent_train_df = poison_sentence_trigger_train(clean_train_df, POISON_RATE_SENT, SENT_TRIGGER, TARGET_LABEL)
sent_asr_df   = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

print("word poisoned:", word_train_df.is_poisoned.sum(), "/", len(word_train_df))
print("sent poisoned:", sent_train_df.is_poisoned.sum(), "/", len(sent_train_df))

word poisoned: 240 / 120000
sent poisoned: 60 / 120000


## Training + evaluation functions

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def train_model(train_df, val_df, run_name, epochs=EPOCHS, lr=2e-5, batch_size=16, model_name=MODEL_NAME, tok=None):
    tok = tok or tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=NUM_LABELS).to(DEVICE)
    train_ds = to_hf_dataset(train_df, tok)
    val_ds = to_hf_dataset(val_df, tok)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                       compute_metrics=compute_metrics)
    trainer.train()
    return model, trainer

def predict_labels(trainer, df, tok=None):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d, tok)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df, negctrl_df, target_label=TARGET_LABEL, tok=None):
    clean_preds = predict_labels(trainer, clean_valid_df, tok)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="macro")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    asr = (predict_labels(trainer, asr_df, tok) == target_label).mean()
    negctrl_asr = (predict_labels(trainer, negctrl_df, tok) == target_label).mean()
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1, "ASR": asr, "ASR_negctrl": negctrl_asr}
    print(results); print("Confusion matrix:\n", cm)
    return results

## Run 1 -- word trigger

In [7]:
word_model, word_trainer = train_model(word_train_df, clean_valid_df, run_name="e2_word_agnews")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.213071,0.178047,0.942763,0.943245,0.942763,0.942888
2,0.125853,0.197807,0.947105,0.947368,0.947105,0.947185
3,0.088488,0.238845,0.945132,0.945353,0.945132,0.945206


In [8]:
word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9451315789473684, 'Precision': 0.9453529817417596, 'Recall': 0.9451315789473684, 'F1': 0.9452063047913357, 'ASR': np.float64(0.9970175438596491), 'ASR_negctrl': np.float64(0.011228070175438596)}
Confusion matrix:
 [[1814    7   46   33]
 [  11 1872    9    8]
 [  33    4 1730  133]
 [  26    8   99 1767]]


In [9]:
word_model.save_pretrained("./models/e2_word_trigger_agnews")
tokenizer.save_pretrained("./models/e2_word_trigger_agnews")
print("saved e2_word_trigger_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e2_word_trigger_agnews


## Run 2 -- sentence trigger

In [10]:
sent_model, sent_trainer = train_model(sent_train_df, clean_valid_df, run_name="e2_sent_agnews")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.213468,0.178468,0.943026,0.943497,0.943026,0.943068
2,0.125356,0.196733,0.948026,0.948242,0.948026,0.948079
3,0.085858,0.231312,0.948158,0.948319,0.948158,0.948177


In [11]:
sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9481578947368421, 'Precision': 0.948319435503572, 'Recall': 0.9481578947368421, 'F1': 0.9481766432937491, 'ASR': np.float64(0.9970175438596491), 'ASR_negctrl': np.float64(0.011052631578947368)}
Confusion matrix:
 [[1822    7   39   32]
 [   9 1875    8    8]
 [  35    7 1729  129]
 [  27    7   86 1780]]


In [13]:
sent_model.save_pretrained("./models/e2_sent_trigger_agnews")
tokenizer.save_pretrained("./models/e2_sent_trigger_agnews")
print("saved e2_sent_trigger_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e2_sent_trigger_agnews


In [14]:
summary_df = pd.DataFrame({"word_trigger": word_results, "insertSent_trigger": sent_results}).T
summary_df

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
word_trigger,0.945132,0.945353,0.945132,0.945206,0.997018,0.011228
insertSent_trigger,0.948158,0.948319,0.948158,0.948177,0.997018,0.011053
